# אתגר: ניתוח טקסט על מדעי הנתונים

בדוגמה זו, בוא נעשה תרגיל פשוט הכולל את כל השלבים של תהליך מסורתי במדעי הנתונים. אין צורך לכתוב קוד, אפשר פשוט ללחוץ על התאים למטה כדי להריץ אותם ולצפות בתוצאה. כאתגר, מומלץ לנסות את הקוד הזה עם נתונים שונים.

## מטרה

בשיעור הזה דנו במושגים שונים הקשורים למדעי הנתונים. בואו ננסה לגלות מושגים נוספים ע"י **כריית טקסט**. נתחיל בטקסט על מדעי הנתונים, נוציא ממנו מילות מפתח, ואז ננסה להמחיש את התוצאה.

בתור טקסט, אני אשתמש בעמוד על מדעי הנתונים מוויקיפדיה:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## שלב 1: קבלת הנתונים

הצעד הראשון בכל תהליך מדעי נתונים הוא קבלת הנתונים. נשתמש בספריית `requests` כדי לעשות זאת:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## שלב 2: המרת הנתונים

השלב הבא הוא להמיר את הנתונים לצורה המתאימה לעיבוד. במקרה שלנו, הורדנו את קוד המקור של ה-HTML מהדף, ואנחנו צריכים להמיר אותו לטקסט פשוט.

ישנן דרכים רבות לעשות זאת. נשתמש ב-[BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), ספריית פייתון פופולרית לניתוח HTML. BeautifulSoup מאפשרת לנו למקד אלמנטים ספציפיים ב-HTML, כך שנוכל להתמקד בתוכן העיקרי של המאמר מוויקיפדיה ולהפחית תפריטי ניווט, סרגלי צד, כותרות תחתונות ותוכן לא רלוונטי אחר (אם כי טקסט קבוע מסוים עלול עדיין להישאר).


ראשית, עלינו להתקין את ספריית BeautifulSoup לניתוח HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## שלב 3: קבלת תובנות

השלב החשוב ביותר הוא להפוך את הנתונים שלנו לצורה שנוכל להפיק ממנה תובנות. במקרה שלנו, אנו רוצים לחלץ מילות מפתח מהטקסט ולראות אילו מילות מפתח משמעותיות יותר.

נשתמש בספריית פייתון שנקראת [RAKE](https://github.com/aneesha/RAKE) לחילוץ מילות מפתח. ראשית, נתקין את הספרייה הזו למקרה שהיא לא מותקנת:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

התפקוד הראשי זמין מהאובייקט `Rake`, שאותו אנו יכולים להתאים באמצעות כמה פרמטרים. במקרה שלנו, נגדיר את האורך המינימלי של מילת מפתח ל-5 תווים, תדירות מינימלית של מילת מפתח במסמך ל-3, ומספר המילים המרבי במילת מפתח - ל-2. אל תהססו לשחק עם ערכים אחרים ולבחון את התוצאה.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


קיבלנו רשימת מונחים יחד עם דרגת חשיבות משויכת. כפי שניתן לראות, התחומים הרלוונטיים ביותר, כגון למידה ממוחשבת ונתונים גדולים, מופיעים ברשימה במיקומים העליונים.

## שלב 4: ויזואליזציה של התוצאה

אנשים יכולים לפרש את הנתונים בצורה הטובה ביותר כאשר הם מוצגים בצורה חזותית. לכן, לעיתים קרובות יש היגיון להציג את הנתונים באופן חזותי כדי להפיק תובנות. נוכל להשתמש בספריית `matplotlib` בפייתון כדי לצייר התפלגות פשוטה של מילות המפתח יחד עם רלוונטיות שלהן:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

עם זאת, ישנה דרך אפילו טובה יותר להמחיש תדירויות מילים - באמצעות **ענן מילים**. נצטרך להתקין ספריה נוספת כדי ליצור את ענן המילים מרשימת מילות המפתח שלנו.


In [ ]:
!{sys.executable} -m pip install wordcloud

אובייקט `WordCloud` אחראי על קבלת טקסט מקורי או רשימה מחושבת מראש של מילים עם התדירויות שלהן, ומחזיר תמונה, שניתן לאחר מכן להציג באמצעות `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

אנו יכולים גם להעביר את הטקסט המקורי ל-`WordCloud` - בוא נראה אם נוכל לקבל תוצאה דומה:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

אתם יכולים לראות שכעת ענן המילים נראה מרשים יותר, אך הוא גם מכיל הרבה רעש (למשל מילים לא קשורות כמו `Retrieved on`). בנוסף, נקבל פחות מילות מפתח המורכבות משתי מילים, כמו *data scientist*, או *computer science*. זאת מכיוון שאלגוריתם RAKE עושה עבודה טובה יותר בבחירת מילות מפתח טובות מהטקסט. דוגמה זו ממחישה את החשיבות של עיבוד מוקדם וניקוי הנתונים, כי תמונה ברורה בסוף תאפשר לנו לקבל החלטות טובות יותר.

בתרגיל זה עברנו תהליך פשוט של חילוץ משמעות מהטקסט של ויקיפדיה, בצורת מילות מפתח וענן מילים. דוגמה זו פשוטה למדי, אך היא מדגימה היטב את כל הצעדים הטיפוסיים שמדען נתונים יבצע כשעובד עם נתונים, החל מרכישת הנתונים, ועד להצגה ויזואלית.

בקורס שלנו נדון בפירוט בכל הצעדים הללו. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
